In [ ]:
import pandas as pd 
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


In [ ]:
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")

train.head()

In [ ]:
# Separate features and target
X = train.drop(columns=["Transported", "PassengerId", "Name"])
y = train["Transported"].astype(int)   # True/False -> 1/0

# Keep PassengerId from test for submission
test_passenger_id = test["PassengerId"]

# Test features (drop same columns)
X_test = test.drop(columns=["PassengerId", "Name"])


In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object", "bool"]).columns

numeric_features, categorical_features


In [ ]:
# Preprocess numeric columns
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Preprocess categorical columns
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# Full model pipeline
model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        random_state=42,
        n_jobs=-1
    ))
])


In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model.fit(X_train, y_train)

y_pred = model.predict(X_valid)
acc = accuracy_score(y_valid, y_pred)
print(f"Validation Accuracy: {acc:.4f}")


In [ ]:
# Fit on all training data
model.fit(X, y)

# Predict probabilities / labels for test
test_pred = model.predict(X_test)

# Convert back to boolean (Kaggle expects True/False)
test_pred_bool = test_pred.astype(bool)

submission = pd.DataFrame({
    "PassengerId": test_passenger_id,
    "Transported": test_pred_bool
})

submission.head()


In [ ]:
submission.to_csv("submission_rf.csv", index=False)
print("Saved submission_rf.csv")
